# 03 - Customer & Sales Analysis
## Repeat Purchase Rate, Sales Growth & State Distribution

**Objective:**
Analyze the macro sales trends, customer purchase frequency, and state-level concentration.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
master = pd.read_csv(PROCESSED_DIR / 'master_analytical_orders.csv')
delivered = master[master['order_status'] == 'delivered'].copy()
delivered['order_purchase_timestamp'] = pd.to_datetime(delivered['order_purchase_timestamp'])
print(f'Delivered orders loaded: {len(delivered):,}')


### 1. Repeat Purchase Rate Calculation


In [ ]:
cust_freq = delivered.groupby('customer_unique_id')['order_id'].nunique()
single_cust = (cust_freq == 1).sum()
repeat_cust = (cust_freq > 1).sum()
total_cust = len(cust_freq)

print(f'Total Unique Customers:     {total_cust:,}')
print(f'Single Order Customers:     {single_cust:,} ({single_cust/total_cust*100:.2f}%)')
print(f'Repeat Customers:           {repeat_cust:,} ({repeat_cust/total_cust*100:.2f}%)')


### 2. Monthly Revenue Trend


In [ ]:
monthly = delivered.groupby(delivered['order_purchase_timestamp'].dt.to_period('M'))['order_revenue'].sum()
monthly.index = monthly.index.astype(str)

plt.figure(figsize=(12, 5))
plt.plot(monthly.index[1:-1], monthly.values[1:-1] / 1e6, marker='o', color='#0284C7', linewidth=2.5)
plt.title('Monthly Sales Revenue (R$ Millions)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue (R$ M)')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


### 3. Top Customer States


In [ ]:
state_rev = delivered.groupby('customer_state')['order_revenue'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=state_rev.index, y=state_rev.values / 1e6, palette='viridis')
plt.title('Top 10 States by Revenue (R$ Millions)', fontsize=14, fontweight='bold')
plt.xlabel('State')
plt.ylabel('Revenue (R$ M)')
plt.show()
